In [2]:
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

# ── Paths ────────────────────────────────────────────────────
SAVE_DIR = "saved_analysis_states"
SCHEMA   = "powerbi"

# ── Postgres connection (same pattern as models_comparison.ipynb) ──
required_vars = ["DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD"]
missing_vars = [var for var in required_vars if os.getenv(var) is None]

if missing_vars:
    raise ValueError(f"Missing environment variables: {missing_vars}")

DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT")
DB_NAME     = os.getenv("DB_NAME")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# ── Test connection + ensure target schema exists ──────────────
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("Connection successful:", result.scalar())

    conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}"))
    conn.commit()
    print(f"Schema ready: {SCHEMA}")

print(f"\nSAVE_DIR : {SAVE_DIR}")
print(f"SCHEMA   : {SCHEMA}")

Connection successful: 1
Schema ready: powerbi

SAVE_DIR : saved_analysis_states
SCHEMA   : powerbi


In [3]:
# ── Load source files from saved_analysis_states/ ──────────────

df_customer_mart = pd.read_parquet(f"{SAVE_DIR}/model_customer_results_mart.parquet")
df_persona       = pd.read_parquet(f"{SAVE_DIR}/cluster_persona_summary.parquet")
df_cluster_perf  = pd.read_parquet(f"{SAVE_DIR}/model_cluster_results.parquet")
df_routing_audit = pd.read_parquet(f"{SAVE_DIR}/routing_audit.parquet")
df_state_table   = pd.read_parquet(f"{SAVE_DIR}/state_table.parquet")
df_assignments   = pd.read_parquet(f"{SAVE_DIR}/customer_cluster_assignments.parquet")

sources = {
    "model_customer_results_mart" : df_customer_mart,
    "cluster_persona_summary"     : df_persona,
    "model_cluster_results"       : df_cluster_perf,
    "routing_audit"               : df_routing_audit,
    "state_table"                 : df_state_table,
    "customer_cluster_assignments": df_assignments,
}

print("=" * 60)
print("SOURCE FILES LOADED")
print("=" * 60)

for name, df in sources.items():
    print(f"\n{name}")
    print(f"  Shape   : {df.shape}")
    print(f"  Columns : {df.columns.tolist()}")

SOURCE FILES LOADED

model_customer_results_mart
  Shape   : (4716, 14)
  Columns : ['customer_id', 'country', 'cluster', 'model_used', 'p_active', 'freq_conditional', 'aov_conditional', 'lgbm_predicted_revenue', 'rfm_predicted_revenue', 'predicted_revenue', 'discounted_clv', 'clv_tier', 'prediction_horizon_days', 'discount_rate']

cluster_persona_summary
  Shape   : (5, 6)
  Columns : ['cluster', 'n_customers', 'pct_of_total', 'persona_name', 'description', 'top_traits_index']

model_cluster_results
  Shape   : (20, 7)
  Columns : ['cluster', 'model', 'n_customers', 'mae', 'rmse', 'top_decile_capture', 'spearman_corr']

routing_audit
  Shape   : (5, 7)
  Columns : ['cluster', 'model_used', 'n_customers', 'mean_predicted_rev', 'median_predicted_rev', 'min_predicted_rev', 'max_predicted_rev']

state_table
  Shape   : (4716, 27)
  Columns : ['customer_id', 'snapshot_date', 'first_purchase_date', 'last_purchase_date', 'frequency', 'repeat_frequency', 'total_revenue', 'avg_order_value', 's

In [4]:
## Building `fact_customers`

Grain: 1 row per customer.

Final columns and their source:

| Column | Source |
|---|---|
| `customer_id` | `model_customer_results_mart` |
| `country` | `model_customer_results_mart` |
| `cluster` | `model_customer_results_mart` |
| `persona_name` | joined from `cluster_persona_summary` on `cluster` |
| `model_used` | `model_customer_results_mart` |
| `predicted_revenue` | `model_customer_results_mart` |
| `discounted_clv` | `model_customer_results_mart` |
| `clv_tier` | `model_customer_results_mart` |
| `p_active` | `model_customer_results_mart` |
| `risk_level` | derived from `p_active` (this cell) |

`risk_level` is a model-output transformation (bucketing a predicted
probability), so it belongs in Python. Business rules that *combine*
`clv_tier` and `risk_level` (e.g. recommended actions) are intentionally
left out — those are built as DAX measures in Power BI.

SyntaxError: invalid character '—' (U+2014) (3047482373.py, line 23)

In [5]:
# ── Join persona_name onto customer mart ────────────────────────

df_fact_customers = df_customer_mart.merge(
    df_persona[["cluster", "persona_name"]],
    on="cluster",
    how="left"
)
# ── Sanity check ─────────────────────────────────────────────
unmatched = df_fact_customers["persona_name"].isna().sum()
print(f"Customers without a matched persona_name: {unmatched}")
assert unmatched == 0, "Some customers did not match a persona — check cluster IDs."

print(f"\nShape after persona join: {df_fact_customers.shape}")
print(f"\nPersona distribution:")
print(df_fact_customers["persona_name"].value_counts())

# ── Derive risk_level from p_active ─────────────────────────────
# p_active = predicted probability the customer remains active.
# Lower p_active => higher churn risk.
#
# Thresholds (documented, adjustable if business feedback says otherwise):
#   p_active >= 0.66        -> "Low"     risk
#   0.33 <= p_active < 0.66 -> "Medium"  risk
#   p_active < 0.33         -> "High"    risk

def assign_risk_level(p):
    if p >= 0.66:
        return "Low"
    elif p >= 0.33:
        return "Medium"
    else:
        return "High"

df_fact_customers["risk_level"] = df_fact_customers["p_active"].apply(assign_risk_level)

# ── Sanity check ─────────────────────────────────────────────
print("Risk level distribution:")
print(df_fact_customers["risk_level"].value_counts())

print("\np_active range per risk_level:")
print(df_fact_customers.groupby("risk_level")["p_active"].agg(["min", "max", "count"]))

null_risk = df_fact_customers["risk_level"].isna().sum()
assert null_risk == 0, f"Null risk_level values detected: {null_risk}"
print("\n[PASS] No nulls in risk_level.")

Customers without a matched persona_name: 0

Shape after persona join: (4716, 15)

Persona distribution:
persona_name
Lapsed One-Time Buyers    1778
Established Customers     1596
Premium Customers          691
At-risk Customers          472
Emerging Customers         179
Name: count, dtype: int64
Risk level distribution:
risk_level
High      2133
Low       1531
Medium    1052
Name: count, dtype: int64

p_active range per risk_level:
                 min       max  count
risk_level                           
High        0.004230  0.329971   2133
Low         0.661066  0.999907   1531
Medium      0.330081  0.659867   1052

[PASS] No nulls in risk_level.


In [6]:
# ── Final column selection for fact_customers ───────────────────

FACT_CUSTOMERS_COLUMNS = [
    "customer_id",
    "country",
    "cluster",
    "persona_name",
    "model_used",
    "predicted_revenue",
    "discounted_clv",
    "clv_tier",
    "p_active",
    "risk_level",
]

fact_customers = df_fact_customers[FACT_CUSTOMERS_COLUMNS].copy()

# ── Null checks on critical columns ─────────────────────────────
critical_cols = ["customer_id", "cluster", "discounted_clv", "clv_tier", "risk_level"]
errors = []

for col in critical_cols:
    n_null = fact_customers[col].isna().sum()
    if n_null > 0:
        errors.append(f"Nulls in '{col}': {n_null}")
    else:
        print(f"  [PASS] No nulls in '{col}'")

# ── Dtype checks ─────────────────────────────────────────────
print(f"\nDtypes:")
print(fact_customers.dtypes)

# clv_tier should be a small fixed category set
expected_tiers = {"Very Low", "Low", "Medium", "High", "VIP"}
actual_tiers = set(fact_customers["clv_tier"].astype(str).unique())
unexpected_tiers = actual_tiers - expected_tiers
if unexpected_tiers:
    errors.append(f"Unexpected clv_tier values: {unexpected_tiers}")
else:
    print(f"\n  [PASS] clv_tier values valid: {actual_tiers}")

# risk_level should only be High/Medium/Low
expected_risk = {"High", "Medium", "Low"}
actual_risk = set(fact_customers["risk_level"].unique())
unexpected_risk = actual_risk - expected_risk
if unexpected_risk:
    errors.append(f"Unexpected risk_level values: {unexpected_risk}")
else:
    print(f"  [PASS] risk_level values valid: {actual_risk}")

# ── Duplicate check ─────────────────────────────────────────────
dupes = fact_customers["customer_id"].duplicated().sum()
if dupes > 0:
    errors.append(f"Duplicate customer_id rows: {dupes}")
else:
    print(f"  [PASS] No duplicate customer_id rows")

if errors:
    for e in errors:
        print(f"  [FAIL] {e}")
    raise AssertionError(f"{len(errors)} validation check(s) failed.")

print(f"\nfact_customers ready: {fact_customers.shape}")
print(fact_customers.head(5).to_string(index=False))

  [PASS] No nulls in 'customer_id'
  [PASS] No nulls in 'cluster'
  [PASS] No nulls in 'discounted_clv'
  [PASS] No nulls in 'clv_tier'
  [PASS] No nulls in 'risk_level'

Dtypes:
customer_id            object
country                object
cluster                 int32
persona_name           object
model_used             object
predicted_revenue     float64
discounted_clv        float64
clv_tier             category
p_active              float64
risk_level             object
dtype: object

  [PASS] clv_tier values valid: {'Medium', 'Very Low', 'High', 'VIP', 'Low'}
  [PASS] risk_level values valid: {'Medium', 'High', 'Low'}
  [PASS] No duplicate customer_id rows

fact_customers ready: (4716, 10)
customer_id        country  cluster           persona_name model_used  predicted_revenue  discounted_clv clv_tier  p_active risk_level
    12346.0 United Kingdom        1  Established Customers   LightGBM        1540.473024     1469.742547      VIP  0.134088       High
    12347.0        Iceland

In [ ]:
## Building `dim_cluster_persona`

Grain: 1 row per cluster.

`cluster_persona_summary.parquet` currently has persona name, size, and
index/lift values relative to the population median — but not the raw
RFM averages the assignment's Page 2 cluster table and radar chart need
(Avg CLV, Avg Frequency, Avg AOV).

These raw averages are recomputed here from `state_table` joined to
`customer_cluster_assignments`, then merged into the persona summary to
produce the final dimension table.

Final columns:

| Column | Source |
|---|---|
| `cluster` | both |
| `persona_name` | `cluster_persona_summary` |
| `description` | `cluster_persona_summary` (if present) |
| `n_customers` | `cluster_persona_summary` |
| `pct_of_total` | `cluster_persona_summary` |
| `avg_frequency` | recomputed from `state_table` |
| `avg_order_value` | recomputed from `state_table` |
| `avg_total_revenue` | recomputed from `state_table` |
| `avg_recency_days` | recomputed from `state_table` |
| `avg_tenure_days` | recomputed from `state_table` |
| `avg_discounted_clv` | recomputed from `fact_customers` |

In [7]:
# ── Recompute raw RFM averages per cluster ──────────────────────

# Join state_table (raw RFM features) to cluster assignments
df_cluster_profile = df_assignments.merge(
    df_state_table,
    on="customer_id",
    how="left"
)

unmatched = df_cluster_profile["frequency"].isna().sum()
print(f"Customers unmatched to state_table: {unmatched}")
assert unmatched == 0, "Some cluster-assigned customers missing from state_table."

RFM_AVG_COLS = {
    "frequency"       : "avg_frequency",
    "avg_order_value" : "avg_order_value",
    "total_revenue"   : "avg_total_revenue",
    "recency_days"    : "avg_recency_days",
    "tenure_days"     : "avg_tenure_days",
}

cluster_rfm_averages = (
    df_cluster_profile
    .groupby("cluster")[list(RFM_AVG_COLS.keys())]
    .mean()
    .rename(columns=RFM_AVG_COLS)
    .round(2)
    .reset_index()
)

# ── Add avg discounted CLV from fact_customers ──────────────────
cluster_clv_avg = (
    fact_customers
    .groupby("cluster")["discounted_clv"]
    .mean()
    .round(2)
    .rename("avg_discounted_clv")
    .reset_index()
)

cluster_rfm_averages = cluster_rfm_averages.merge(cluster_clv_avg, on="cluster", how="left")

print("Cluster RFM averages:")
print(cluster_rfm_averages.to_string(index=False))

Customers unmatched to state_table: 0
Cluster RFM averages:
 cluster  avg_frequency  avg_order_value  avg_total_revenue  avg_recency_days  avg_tenure_days  avg_discounted_clv
       0           1.14           362.08             435.40             14.08            15.18              415.41
       1           6.19           408.69            2462.71            128.69           369.74              583.13
       2           2.33           292.86             662.30            130.80           383.83              185.22
       3          14.11           471.76            8129.87             11.95           376.50             3034.93
       4           1.24           321.83             380.79            234.41           243.68              114.24


In [8]:
# ── Merge raw averages into persona summary ─────────────────────

persona_cols_available = [c for c in ["cluster", "persona_name", "description",
                                        "n_customers", "pct_of_total"]
                           if c in df_persona.columns]

dim_cluster_persona = df_persona[persona_cols_available].merge(
    cluster_rfm_averages,
    on="cluster",
    how="left"
)

# ── Validation ───────────────────────────────────────────────
n_clusters_expected = dim_cluster_persona["cluster"].nunique()
print(f"Clusters in dim_cluster_persona: {n_clusters_expected}")

null_counts = dim_cluster_persona.isna().sum()
nulls_present = null_counts[null_counts > 0]
if len(nulls_present) > 0:
    print(f"\n[WARNING] Nulls detected:\n{nulls_present}")
else:
    print(f"\n[PASS] No nulls in dim_cluster_persona")

print(f"\ndim_cluster_persona ready: {dim_cluster_persona.shape}")
print(dim_cluster_persona.to_string(index=False))

Clusters in dim_cluster_persona: 5

[PASS] No nulls in dim_cluster_persona

dim_cluster_persona ready: (5, 11)
 cluster           persona_name                                                                                                                                                                  description  n_customers  pct_of_total  avg_frequency  avg_order_value  avg_total_revenue  avg_recency_days  avg_tenure_days  avg_discounted_clv
       0     Emerging Customers      Brand-new customers (lowest tenure, very recent first purchase). Revenue is still low simply due to short history, not low value -- too early to judge long-term worth.          179           3.8           1.14           362.08             435.40             14.08            15.18              415.41
       1  Established Customers Largest high-value segment: strong frequency, revenue, and tenure, but recency is near the population median and most are flagged dormant -- engagement is starting to fade.         

In [ ]:
## Building `fact_model_performance`

Grain: 1 row per cluster × model.

This is a cleaned export of `model_cluster_results.parquet` — no new
computation needed, just column renaming for Power BI friendliness and
a validation pass.

## Building `dim_model_routing`

Grain: 1 row per cluster × model_used.

This is a cleaned export of `routing_audit.parquet` — shows which model
was deployed for each cluster and the resulting prediction stats, used
for Page 4's "Model Routing Breakdown" table.

In [9]:
# ── Clean fact_model_performance ─────────────────────────────────

fact_model_performance = df_cluster_perf.rename(columns={
    "n_customers"        : "n_customers",
    "mae"                : "mae",
    "rmse"                : "rmse",
    "top_decile_capture" : "top_decile_capture",
    "spearman_corr"      : "spearman_corr",
}).copy()

EXPECTED_COLS = ["cluster", "model", "n_customers", "mae", "rmse",
                  "top_decile_capture", "spearman_corr"]

missing_cols = [c for c in EXPECTED_COLS if c not in fact_model_performance.columns]
assert not missing_cols, f"Missing expected columns: {missing_cols}"

fact_model_performance = fact_model_performance[EXPECTED_COLS]

# ── Validation ───────────────────────────────────────────────
null_counts = fact_model_performance.isna().sum()
nulls_present = null_counts[null_counts > 0]
if len(nulls_present) > 0:
    print(f"[WARNING] Nulls detected:\n{nulls_present}")
else:
    print("[PASS] No nulls in fact_model_performance")

print(f"\nfact_model_performance ready: {fact_model_performance.shape}")
print(fact_model_performance.to_string(index=False))

[WARNING] Nulls detected:
mae              3
rmse             3
spearman_corr    3
dtype: int64

fact_model_performance ready: (20, 7)
 cluster    model  n_customers         mae        rmse  top_decile_capture  spearman_corr
       0 LightGBM           31  511.947909  764.298633            0.500000       0.380706
       0      RFM           31  348.076774  486.615713            0.500000       0.411145
       0   BG-NBD           31         NaN         NaN            0.000000            NaN
       0   Pareto           31  744.297173 1983.751446            0.500000       0.351189
       1 LightGBM          317  514.542324  900.881367            0.343750       0.511412
       1      RFM          317 1781.412524 2939.835778            0.437500       0.470616
       1   BG-NBD          317         NaN         NaN            0.406250            NaN
       1   Pareto          317  767.066237 1255.617488            0.406250       0.462391
       2 LightGBM           84  221.892918  351.315886 

In [10]:
# ── Clean dim_model_routing ──────────────────────────────────────

dim_model_routing = df_routing_audit.rename(columns={
    "model_used"            : "model_used",
    "n_customers"           : "n_customers",
    "mean_predicted_rev"    : "mean_predicted_revenue",
    "median_predicted_rev"  : "median_predicted_revenue",
    "min_predicted_rev"     : "min_predicted_revenue",
    "max_predicted_rev"     : "max_predicted_revenue",
}).copy()

EXPECTED_COLS = ["cluster", "model_used", "n_customers",
                  "mean_predicted_revenue", "median_predicted_revenue",
                  "min_predicted_revenue", "max_predicted_revenue"]

missing_cols = [c for c in EXPECTED_COLS if c not in dim_model_routing.columns]
assert not missing_cols, f"Missing expected columns: {missing_cols}"

dim_model_routing = dim_model_routing[EXPECTED_COLS]

# ── Validation ───────────────────────────────────────────────
null_counts = dim_model_routing.isna().sum()
nulls_present = null_counts[null_counts > 0]
if len(nulls_present) > 0:
    print(f"[WARNING] Nulls detected:\n{nulls_present}")
else:
    print("[PASS] No nulls in dim_model_routing")

print(f"\ndim_model_routing ready: {dim_model_routing.shape}")
print(dim_model_routing.to_string(index=False))

[PASS] No nulls in dim_model_routing

dim_model_routing ready: (5, 7)
 cluster model_used  n_customers  mean_predicted_revenue  median_predicted_revenue  min_predicted_revenue  max_predicted_revenue
       0        RFM          179              435.401955                300.360000                  17.55            5936.800000
       1   LightGBM         1596              611.195286                355.914801                   0.00           15535.811576
       2   LightGBM          472              194.129699                129.447283                   0.00            2780.966377
       3   LightGBM          691             3180.986802               1139.175519                   0.00          235640.789581
       4   LightGBM         1778              119.740645                 70.195869                   0.00            1965.007639


In [ ]:
## Pre-Write Validation Checklist

Before writing to Postgres, confirm across all four tables:

1. **Row counts match expectations**
   - `fact_customers` ≈ 4,339 customers (matches `model_customer_results_mart`)
   - `dim_cluster_persona` = number of clusters (5)
   - `fact_model_performance` = clusters × models evaluated
   - `dim_model_routing` = clusters (1 model per cluster, per routing logic)

2. **No nulls in key columns** across all four tables (already checked
   individually per-cell, re-verified here together).

3. **Foreign key consistency** — every `cluster` value in `fact_customers`
   must exist in `dim_cluster_persona`, `fact_model_performance`, and
   `dim_model_routing`. No orphaned cluster IDs in either direction.

4. **Category integrity** — `clv_tier` and `risk_level` in `fact_customers`
   only contain the expected fixed value sets (checked in Cell 7, re-verified
   here as a final gate before write).

5. **No duplicate grain** — `fact_customers` has no duplicate `customer_id`;
   `dim_cluster_persona` has no duplicate `cluster`.

In [11]:
# ── Final cross-table validation before writing to Postgres ─────

errors = []

# 1. Row count sanity
print("=" * 60)
print("ROW COUNTS")
print("=" * 60)
print(f"  fact_customers          : {len(fact_customers)}")
print(f"  dim_cluster_persona     : {len(dim_cluster_persona)}")
print(f"  fact_model_performance  : {len(fact_model_performance)}")
print(f"  dim_model_routing       : {len(dim_model_routing)}")

if len(fact_customers) == 0:
    errors.append("fact_customers is empty")

# 2. Null checks (re-verify together)
print("\n" + "=" * 60)
print("NULL CHECKS")
print("=" * 60)
tables = {
    "fact_customers"         : fact_customers,
    "dim_cluster_persona"    : dim_cluster_persona,
    "fact_model_performance" : fact_model_performance,
    "dim_model_routing"      : dim_model_routing,
}
for name, df in tables.items():
    null_total = df.isna().sum().sum()
    status = "PASS" if null_total == 0 else "WARN"
    print(f"  [{status}] {name}: {null_total} total nulls")

# 3. Foreign key consistency on cluster
print("\n" + "=" * 60)
print("FOREIGN KEY CONSISTENCY (cluster)")
print("=" * 60)

fc_clusters     = set(fact_customers["cluster"].unique())
persona_clusters = set(dim_cluster_persona["cluster"].unique())
perf_clusters    = set(fact_model_performance["cluster"].unique())
routing_clusters = set(dim_model_routing["cluster"].unique())

orphans_persona = fc_clusters - persona_clusters
orphans_perf    = fc_clusters - perf_clusters
orphans_routing = fc_clusters - routing_clusters

if orphans_persona:
    errors.append(f"Clusters in fact_customers missing from dim_cluster_persona: {orphans_persona}")
else:
    print("  [PASS] fact_customers clusters all present in dim_cluster_persona")

if orphans_perf:
    errors.append(f"Clusters in fact_customers missing from fact_model_performance: {orphans_perf}")
else:
    print("  [PASS] fact_customers clusters all present in fact_model_performance")

if orphans_routing:
    errors.append(f"Clusters in fact_customers missing from dim_model_routing: {orphans_routing}")
else:
    print("  [PASS] fact_customers clusters all present in dim_model_routing")

# 4. Category integrity (re-verify)
print("\n" + "=" * 60)
print("CATEGORY INTEGRITY")
print("=" * 60)

expected_tiers = {"Very Low", "Low", "Medium", "High", "VIP"}
actual_tiers = set(fact_customers["clv_tier"].astype(str).unique())
if not actual_tiers.issubset(expected_tiers):
    errors.append(f"Unexpected clv_tier values: {actual_tiers - expected_tiers}")
else:
    print(f"  [PASS] clv_tier values: {actual_tiers}")

expected_risk = {"High", "Medium", "Low"}
actual_risk = set(fact_customers["risk_level"].unique())
if not actual_risk.issubset(expected_risk):
    errors.append(f"Unexpected risk_level values: {actual_risk - expected_risk}")
else:
    print(f"  [PASS] risk_level values: {actual_risk}")

# 5. Duplicate grain checks
print("\n" + "=" * 60)
print("DUPLICATE GRAIN CHECKS")
print("=" * 60)

dupe_customers = fact_customers["customer_id"].duplicated().sum()
dupe_clusters  = dim_cluster_persona["cluster"].duplicated().sum()

if dupe_customers > 0:
    errors.append(f"Duplicate customer_id in fact_customers: {dupe_customers}")
else:
    print("  [PASS] No duplicate customer_id in fact_customers")

if dupe_clusters > 0:
    errors.append(f"Duplicate cluster in dim_cluster_persona: {dupe_clusters}")
else:
    print("  [PASS] No duplicate cluster in dim_cluster_persona")

# ── Final result ─────────────────────────────────────────────
print("\n" + "=" * 60)
if errors:
    for e in errors:
        print(f"  [FAIL] {e}")
    raise AssertionError(f"{len(errors)} validation check(s) failed. Fix before writing to Postgres.")
else:
    print("  ALL VALIDATION CHECKS PASSED — ready to write to Postgres.")
print("=" * 60)

ROW COUNTS
  fact_customers          : 4716
  dim_cluster_persona     : 5
  fact_model_performance  : 20
  dim_model_routing       : 5

NULL CHECKS
  [PASS] fact_customers: 0 total nulls
  [PASS] dim_cluster_persona: 0 total nulls
  [WARN] fact_model_performance: 9 total nulls
  [PASS] dim_model_routing: 0 total nulls

FOREIGN KEY CONSISTENCY (cluster)
  [PASS] fact_customers clusters all present in dim_cluster_persona
  [PASS] fact_customers clusters all present in fact_model_performance
  [PASS] fact_customers clusters all present in dim_model_routing

CATEGORY INTEGRITY
  [PASS] clv_tier values: {'Medium', 'Very Low', 'High', 'VIP', 'Low'}
  [PASS] risk_level values: {'Medium', 'High', 'Low'}

DUPLICATE GRAIN CHECKS
  [PASS] No duplicate customer_id in fact_customers
  [PASS] No duplicate cluster in dim_cluster_persona

  ALL VALIDATION CHECKS PASSED — ready to write to Postgres.


In [ ]:
## Write Plan

Four tables, written to the `powerbi` schema, `replace` mode (this notebook
represents the latest full snapshot, not an incremental/historical load):

| DataFrame | Postgres table | Write mode |
|---|---|---|
| `fact_customers` | `powerbi.fact_customers` | replace |
| `dim_cluster_persona` | `powerbi.dim_cluster_persona` | replace |
| `fact_model_performance` | `powerbi.fact_model_performance` | replace |
| `dim_model_routing` | `powerbi.dim_model_routing` | replace |

Note: `fact_model_performance` has 9 known nulls in `spearman_corr` for
cluster-model combinations with too few customers to compute a valid
Spearman correlation (n < 3). This is expected and will be re-confirmed
post-write rather than masked here.

In [12]:
# ── Write all four tables to Postgres ────────────────────────────

tables_to_write = {
    "fact_customers"         : fact_customers,
    "dim_cluster_persona"    : dim_cluster_persona,
    "fact_model_performance" : fact_model_performance,
    "dim_model_routing"      : dim_model_routing,
}

write_summary = []

for table_name, df in tables_to_write.items():
    try:
        df.to_sql(
            name=table_name,
            con=engine,
            schema=SCHEMA,
            if_exists="replace",
            index=False
        )
        write_summary.append((table_name, len(df), "SUCCESS"))
        print(f"  [OK] {SCHEMA}.{table_name} — {len(df)} rows written")
    except Exception as e:
        write_summary.append((table_name, len(df), f"FAILED: {e}"))
        print(f"  [FAIL] {SCHEMA}.{table_name} — {e}")

print("\n" + "=" * 60)
print("WRITE SUMMARY")
print("=" * 60)
for name, n_rows, status in write_summary:
    print(f"  {name:<28} {n_rows:>6} rows   {status}")

failures = [s for _, _, s in write_summary if s != "SUCCESS"]
if failures:
    raise RuntimeError(f"{len(failures)} table write(s) failed — see summary above.")

print("\nAll tables written successfully.")

  [OK] powerbi.fact_customers — 4716 rows written
  [OK] powerbi.dim_cluster_persona — 5 rows written
  [OK] powerbi.fact_model_performance — 20 rows written
  [OK] powerbi.dim_model_routing — 5 rows written

WRITE SUMMARY
  fact_customers                 4716 rows   SUCCESS
  dim_cluster_persona               5 rows   SUCCESS
  fact_model_performance           20 rows   SUCCESS
  dim_model_routing                 5 rows   SUCCESS

All tables written successfully.


In [ ]:
## Summary & Power BI Connection Notes

### Tables available in Postgres (`powerbi` schema)

| Table | Grain | Row count |
|---|---|---|
| `powerbi.fact_customers` | 1 row per customer | 4,716 |
| `powerbi.dim_cluster_persona` | 1 row per cluster | 5 |
| `powerbi.fact_model_performance` | 1 row per cluster × model | 20 |
| `powerbi.dim_model_routing` | 1 row per cluster × model_used | 5 |

### Data model relationships (mirror this in Power BI's Model view)

- `fact_customers.cluster` → `dim_cluster_persona.cluster` (many-to-one)
- `fact_customers.cluster` → `fact_model_performance.cluster` (many-to-one,
  but note `fact_model_performance` is itself cluster × model — treat it as
  a second fact table joined on `cluster`, not a dimension)
- `fact_customers.cluster` → `dim_model_routing.cluster` (many-to-one)

### What is intentionally NOT in these tables

`recommended_action`, `revenue_at_risk`, and `VIP_customer_flag` are not
precomputed. They are built as DAX measures in Power BI from
`fact_customers.clv_tier`, `fact_customers.risk_level`, and
`fact_customers.discounted_clv`. This is a deliberate architecture choice:
Python/Postgres own the modeling layer (CLV prediction, clustering,
persona assignment, risk derivation); Power BI owns the business logic
layer (executive KPIs, recommended actions, interactive drill-through).

### Known data caveat

`fact_model_performance.spearman_corr` contains 9 nulls — these are
cluster-model combinations where the test-set slice had fewer than 3
customers, so Spearman correlation is mathematically undefined (not a
data quality bug). Handle with `COALESCE` or a blank-aware visual in
Power BI rather than filtering these rows out, since the MAE/RMSE values
for those rows are still valid.

### Next step

Connect Power BI Desktop to Postgres (`Get Data → PostgreSQL database`),
point at `DB_HOST`/`DB_NAME`, schema `powerbi`, and import the four tables
above as the data model foundation for the four-page report.